# Kiến trúc Transformer Encoder hoàn chỉnh bằng PyTorch (Gộp 2 lớp Attention)

Chào mừng bạn đến với hướng dẫn lập trình chi tiết cấu phần **Encoder (Bộ mã hóa)** trong mô hình Transformer. 

Theo yêu cầu tối ưu hóa cấu trúc mã nguồn, chúng ta sẽ gộp hoàn toàn cơ chế **Scaled Dot-Product Attention** vào bên trong lớp **MultiHeadAttention** duy nhất. Điều này giúp code gọn gàng, giảm thiểu các lớp phụ trợ và dễ dàng bảo trì.

Để dựng hoàn chỉnh một **Transformer Encoder**, chúng ta cần xây dựng các thành phần sau:
1. **Input Embedding**: Biến đổi các Token ID thành vector liên tục.
2. **Positional Encoding**: Thêm thông tin vị trí của từ vào vector nhúng.
3. **Multi-Head Self-Attention (Merged)**: Cơ chế chú ý đa đầu tích hợp sẵn phép tính scaled dot-product.
4. **Position-wise Feed-Forward Network (FFN)**: Lớp mạng nơ-ron truyền thẳng độc lập cho từng vị trí từ.
5. **Layer Normalization & Residual Connection (Add & Norm)**: Đảm bảo độ ổn định huấn luyện mô hình sâu.
6. **Encoder Layer**: Khối chứa 1 khối MHA gộp và 1 khối FFN kèm các liên kết tắt và chuẩn hóa.
7. **Stack of Encoders**: Xếp chồng $N$ khối Encoder Layer liên tiếp.

---

## 1. Lớp Mã Hóa Vị Trí (Positional Encoding)

Sử dụng các hàm Sine và Cosine ở các tần số khác nhau để tạo ra các vector vị trí:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [13]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    """
    Lớp chèn vector vị trí (Positional Encoding) sử dụng hàm hình sin.
    """
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## 2. Khối Multi-Head Attention tích hợp cơ chế Scaled Dot-Product

Lớp **MultiHeadAttention** bên dưới tự thực hiện phép nhân Query-Key, chia tỉ lệ $\sqrt{d_k}$, áp dụng Mask, tính toán Softmax và nhân với Value mà không cần gọi thêm lớp bên ngoài.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Lớp Multi-Head Self-Attention tích hợp trực tiếp cơ chế Scaled Dot-Product Attention bên trong.
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model phải chia hết cho num_heads!"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Lớp chiếu tuyến tính tuyến cho Q, K, V
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # Lớp chiếu tuyến tính cuối cùng
        self.out_linear = nn.Linear(d_model, d_model)
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Chiếu tuyến tính
        q_proj = self.q_linear(q)  # q_proj = q * W_Q + b_Q
        k_proj = self.k_linear(k)  # k_proj = k * W_K + b_K
        v_proj = self.v_linear(v)  # v_proj = v * W_V + b_V

        
        # 2. Tách nhỏ thành nhiều attention head  song song (Split Heads)
        
        # Thay vì biểu diễn mỗi từ bằng một vector dài nâng cao d_model chiều, chúng ta cắt vector d_model chiều đó thành num_heads khúc nhỏ, mỗi khúc dài d_model / num_heads chiều (tương ứng với num_heads attention head).
        
        # Tại sao bước hoán đổi chiều này lại vô cùng quan trọng?
        # Trong PyTorch, khi thực hiện phép nhân ma trận chập (torch.matmul) trên các Tensor nhiều chiều, phép nhân sẽ mặc định được áp dụng lên hai chiều cuối cùng. Các chiều đứng trước sẽ được coi như các "chiều lô" (batch dimensions) để chạy song song.
        # Bằng cách đưa num_heads lên chiều thứ hai, Tensor của chúng ta có dạng [batch_size, num_heads, seq_len, d_k]. Lúc này:
        # Chiều batch_size  và chiều num_heads  đóng vai trò là chiều lô.
        # Hai chiều cuối cùng là [seq_len, d_k] sẽ trực tiếp tham gia vào phép tính nhân ma trận Attention chéo.
        q_split = q_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k_split = k_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v_split = v_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Tính toán Scaled Dot-Product Attention trực tiếp
        # scores: [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(q_split, k_split.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Áp dụng Mask (nếu có)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # Trọng số attention Softmax
        attn_probs = F.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        
        # Tổng chập thông tin với Value
        attn_out = torch.matmul(attn_probs, v_split)
        
        # 4. Hợp nhất các head lại với nhau (Concatenate Heads)
        # Lệnh .contiguous() sẽ ép PyTorch copy và sắp xếp lại toàn bộ các phần tử vật lý nằm liên tục đúng theo trật tự mới [batch_size, seq_len, num_heads, d_model/num_heads].

        attn_concat = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model) # [batch_size, seq_len, d_model]
        # Khi chúng ta ghép nối (concatenate), chúng ta chỉ đang xếp các kết quả này nằm cạnh nhau trong một vector d_model chiều.
        # Lúc này, chưa hề có sự giao tiếp hay liên kết thông tin chéo giữa các đầu với nhau. Chúng giống như những báo cáo độc lập được kẹp chung vào một tập hồ sơ.
        # 5. Chiếu tuyến tính đầu ra cuối cùng W_O
        # Lớp tuyến tính self.out_linear chính là "Người tổng hợp". Nhiệm vụ của nó là nhân ma trận trọng số $W_O$ để phối trộn, kết hợp và đan xen các nguồn thông tin từ cả 8 đầu chú ý lại với nhau, tạo nên một biểu diễn ngữ cảnh thống nhất, sâu sắc và toàn diện nhất.

        output = self.out_linear(attn_concat) 
        
        return output, attn_probs

## 3. Khối Nơ-ron Truyền Thẳng Theo Vị Trí (Position-wise Feed-Forward Network - FFN)

Công thức của FFN:
$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

In [15]:
class PositionwiseFeedForward(nn.Module):
    """
    Lớp mạng nơ-ron truyền thẳng tuyến tính áp dụng độc lập cho từng vị trí.
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

## 4. Khối Lớp Bộ Mã Hóa (Encoder Layer)

Kết hợp **Multi-Head Self-Attention (Gộp)** và **Position-wise FFN** kèm Residual Connection và Layer Normalization.

In [16]:
class EncoderLayer(nn.Module):
    """
    Lớp con cấu thành của Encoder (chứa Multi-Head Attention và Position-wise Feed-Forward).
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # 1. Nhánh 1: Multi-Head Self-Attention + Residual + LayerNorm
        attn_out, _ = self.mha(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # 2. Nhánh 2: Position-wise FFN + Residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x

## 5. Kiến trúc Transformer Encoder hoàn chỉnh

Xếp chồng tuần tự $N$ lớp `EncoderLayer`.

In [17]:
class TransformerEncoder(nn.Module):
    """
    Hợp phần Transformer Encoder hoàn chỉnh xếp chồng N lớp Encoder Layers.
    """
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, 
                 num_heads: int, d_ff: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=max_len, dropout=dropout)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model=d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None):
        # 1. Embeddings từ Token ID
        out = self.token_embedding(x)
        
        # 2. Cộng thêm vector thông tin vị trí
        out = self.positional_encoding(out)
        
        # 3. Đi lần lượt qua tuần tự N lớp EncoderLayer
        for layer in self.layers:
            out = layer(out, mask=mask)
            
        # 4. Chuẩn hóa Layer Norm cuối cùng
        out = self.norm(out)
        
        return out

## 6. Chạy thử nghiệm thực tế & Kiểm chứng chiều Tensor (Shapes)

In [19]:
# Thiết lập siêu tham số
vocab_size = 1000
batch_size = 2
seq_len = 6
num_layers = 6
d_model = 128
num_heads = 8
d_ff = 512

# 1. Khởi tạo mô hình
encoder = TransformerEncoder(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=0.1
)
encoder.eval()

# 2. Giả lập Tensor Token ID đầu vào: kích thước [batch_size, seq_len]
X = torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len))
print(f"[ĐẦU VÀO] Token IDs nguyên bản (X):\n{X}\nShape: {X.shape}")


# 4. Chạy mô hình qua Encoder
with torch.no_grad():
    encoder_output = encoder(X)

print("\n" + "="*40 + " KẾT QUẢ ĐẦU RA " + "="*40)
print(f"* Shape của Encoder Output: {encoder_output.shape} -> [batch_size, seq_len, d_model]")


[ĐẦU VÀO] Token IDs nguyên bản (X):
tensor([[223, 897, 868,  60,  75, 233],
        [984, 656, 275, 282, 907, 303]])
Shape: torch.Size([2, 6])

======================================== KẾT QUẢ ĐẦU RA ========================================
* Shape của Encoder Output: torch.Size([2, 6, 128]) -> [batch_size, seq_len, d_model]
